In [2]:
import pandas as pd
from pulp import LpMaximize, LpProblem, LpVariable, lpSum

import pickle

with open('shop_dataset.pkl', 'rb') as f:
    data = pickle.load(f)


data.head()


,Product_Name,Category,Store_ID,Purchase_Date,Sales_Date,Quantity_Purchased,Quantity_Sold,Purchase_Price,Selling_Price,Discount,Revenue,Stock_Level,Demand_Forecast,Region,Competitor_Price,Marketing_Campaign,Seasonality,Returns
0,Eggs,Snacks,4,2022-01-01 00:00:00.000000000,2022-12-09 21:05:27.272727276,37,1,3.20,12.30,0.14,10.5780,78,16,South,8.19,0,Summer Sales,1
1,Juice,Bakery,6,2022-01-04 16:29:05.454545454,2022-05-21 02:25:27.272727274,45,27,6.07,5.83,0.27,114.9093,9,56,East,17.41,1,Holiday Season,1
2,Cookies,Beverages,7,2022-01-08 08:58:10.909090909,2022-08-13 21:34:32.727272728,14,36,14.71,3.60,0.26,95.9040,6,75,South,12.63,0,Black Friday,1
3,Bread,Snacks,3,2022-01-12 01:27:16.363636363,2022-08-17 14:03:38.181818184,18,34,4.50,29.22,0.06,933.8712,58,97,East,19.84,1,Holiday Season,2
4,Cheese,Bakery,5,2022-01-15 17:56:21.818181818,2022-12-17 06:03:38.181818184,33,42,14.10,7.93,0.08,306.4152,81,70,West,2.90,0,Holiday Season,1


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Product_Name        100 non-null    object        
 1   Category            100 non-null    object        
 2   Store_ID            100 non-null    int32         
 3   Purchase_Date       100 non-null    datetime64[ns]
 4   Sales_Date          100 non-null    datetime64[ns]
 5   Quantity_Purchased  100 non-null    int32         
 6   Quantity_Sold       100 non-null    int32         
 7   Purchase_Price      100 non-null    float64       
 8   Selling_Price       100 non-null    float64       
 9   Discount            100 non-null    float64       
 10  Revenue             100 non-null    float64       
 11  Stock_Level         100 non-null    int32         
 12  Demand_Forecast     100 non-null    int32         
 13  Region              100 non-null    object        


In [5]:
budget = 5000  
model = LpProblem("Procurement_Optimization", LpMaximize)

purchase_quantities = {i: LpVariable(f"purchase_qty_{i}", lowBound=0, cat='Integer') for i in data.index}

profit = lpSum((data.loc[i, 'Selling_Price'] * (1 - data.loc[i, 'Discount']) - data.loc[i, 'Purchase_Price']) 
               * purchase_quantities[i] for i in data.index)
model += profit, "Total_Profit"

model += lpSum(data.loc[i, 'Purchase_Price'] * purchase_quantities[i] for i in data.index) <= budget, "Budget_Constraint"

for i in data.index:
    model += purchase_quantities[i] + data.loc[i, 'Stock_Level'] <= data.loc[i, 'Demand_Forecast'], f"Demand_Constraint_{i}"

model.solve()

print("Статус решения:", model.status)
print("Оптимальное количество закупок для каждого товара:")

for i in data.index:
    print(f"{data.loc[i, 'Product_Name']}: Закупка {purchase_quantities[i].value()}")

print("Ожидаемая прибыль:", model.objective.value())

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pulp/solverdir/cbc/osx/64/cbc /var/folders/hk/tknfq7v96pqc7dqsct5qcdt40000gn/T/881b83519dfe4d9abd85fc1c6f356946-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /var/folders/hk/tknfq7v96pqc7dqsct5qcdt40000gn/T/881b83519dfe4d9abd85fc1c6f356946-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 106 COLUMNS
At line 607 RHS
At line 709 BOUNDS
At line 810 ENDATA
Problem MODEL has 101 rows, 100 columns and 200 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Problem is infeasible - 0.00 seconds
Option for printingOptions changed from normal to all
Total time (CPU seconds):       0.00   (Wallclock seconds):       0.00

Статус решения: -1
Оптимальное количество закупок для каждого товара:
Eggs: Закупка -62.0 единиц
Juice: Закупка 0

In [ ]:
def knapsack(capacity, weights, values, n):
    if n == 0 or capacity == 0:
        return 0
    if weights[n-1] > capacity:
        return knapsack(capacity, weights, values, n-1)
    else:
        return max(
            values[n-1] + knapsack(capacity - weights[n-1], weights, values, n-1),
            knapsack(capacity, weights, values, n-1)
        )
weights = data['Purchase_Price']
values = (data['Selling_Price'] * (1 - data['Discount']) - data['Purchase_Price']).tolist()
capacity = budget

profit_knapsack = knapsack(capacity, weights, values, len(data))
print( profit_knapsack)
